# PA4 Task 1 — Kaggle runner (pruning + weight-only quantization of Qwen2.5-1.5B)

One Kaggle version does everything (**Save Version → Save & Run All (Commit)**, runs headless, ≤ 12 h):

| Phase | GPU 0 | GPU 1 |
|---|---|---|
| setup | data materialisation (`setup_data.py --public`), model download, **preflight** | – |
| main | `family_prune`: env → FP16 baseline → P1 ×3 → P2 ×3 → 2:4 → P3 sensitivity → P3 dev selection (frozen) → P3 ×2 → kernel microbench → post-hoc transfer | `family_quant`: AWQ scale search → Q1 ×3 → Q2 ×3 → Q3 sensitivity → Q3 dev selection (frozen) → Q3 ×2 → post-hoc transfer |
| end | report (tables + figures), zip | – |

Every step is idempotent: if the session dies, add this notebook's output as input, set `PREV_OUTPUT`, and run again;
finished conditions are skipped. With a single GPU the two families run one after the other.

Notebook settings: **Accelerator = GPU T4 ×2**, **Internet = On**.

In [ ]:
# ---------------- settings (edit these) ----------------
REPO_URL      = "https://github.com/wasaymj/ATML.git"
BRANCH        = "main"
PREV_OUTPUT   = ""        # e.g. "/kaggle/input/pa4-task1-runner" to resume (Add Input -> Your Work -> this notebook)
RUN_PREFLIGHT = True      # ~5-10 min of correctness checks on the real model before the long runs

import os, subprocess, sys, time, shutil, glob, json
WORK = "/kaggle/working"
REPO = f"{WORK}/ATML"
PA4  = f"{REPO}/PA4"
TASK = f"{PA4}/Task 1"
DATA = f"{PA4}/atml-pa4-datasets"
LOGS = f"{WORK}/logs"
os.makedirs(LOGS, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print(shutil.disk_usage(WORK))

In [ ]:
# ---------------- get the code ----------------
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ---------------- pinned dependencies (torch stays Kaggle's CUDA build) ----------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TASK}/requirements.txt"], check=True)
print(subprocess.run([sys.executable, "-c",
    "import torch, transformers, huggingface_hub, datasets; print('torch', torch.__version__, 'cuda', torch.version.cuda, "
    "'| transformers', transformers.__version__, '| hub', huggingface_hub.__version__, '| datasets', datasets.__version__)"],
    capture_output=True, text=True).stdout)

In [ ]:
# ---------------- restore a previous version's outputs (resume only) ----------------
def copy_tree(src, dst):
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True); print("restored", src, "->", dst)
if PREV_OUTPUT:
    copy_tree(f"{PREV_OUTPUT}/ATML/PA4/Task 1/results", f"{TASK}/results")
    copy_tree(f"{PREV_OUTPUT}/ATML/PA4/atml-pa4-datasets/data", f"{DATA}/data")

## Data: fixed public selections (C4 calibration/development, WikiText-2 and C4 windows, HellaSwag IDs)
`setup_data.py --public` resolves each Hugging Face repo to a commit SHA on its first run and writes
`data/local_source_revisions.json`. **Download that file afterwards and commit it** (it is also copied to
`results/data_selection/`) so every later run uses the same commits. C4 streaming takes a few minutes.

In [ ]:
lock = f"{DATA}/data/local_source_revisions.json"
print("revision lock present before setup:", os.path.exists(lock))
subprocess.run([sys.executable, "scripts/setup_data.py", "--public"], cwd=DATA, check=True)
subprocess.run([sys.executable, "scripts/check_setup.py"], cwd=DATA)          # prefix banks are optional (Task 3)
print(open(lock).read())

In [ ]:
# ---------------- download the exact model revision once (both GPU processes then read the local cache) ----------------
from huggingface_hub import snapshot_download
rev = json.load(open(lock))["revisions"]["model_base"]
path = snapshot_download("Qwen/Qwen2.5-1.5B", revision=rev)
print("Qwen/Qwen2.5-1.5B @", rev, "->", path)

In [ ]:
# ---------------- helpers ----------------
ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false",
           HF_HUB_DISABLE_PROGRESS_BARS="1")

def run(step, gpu=0):
    cmd = [sys.executable, "scripts/run_task1.py", "--step", step]
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=TASK, env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))).returncode

def launch(name, gpu, steps):
    script = f"{LOGS}/{name}.sh"
    lines = [f'cd "{TASK}"'] + [f'{sys.executable} scripts/run_task1.py --step {s} || echo "STEP FAILED: {s}"' for s in steps]
    open(script, "w").write("\n".join(lines) + "\n")
    print(f"[{name}] GPU {gpu}: {steps} -> log {LOGS}/{name}.log", flush=True)
    return subprocess.Popen(["bash", script], stdout=open(f"{LOGS}/{name}.log", "a"), stderr=subprocess.STDOUT,
                            env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)))

def wait(procs, every_min=10):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every_min * 60)
        for name in procs:
            tail = open(f"{LOGS}/{name}.log", errors="replace").read().splitlines()[-3:]
            print(f"--- {name} ({time.strftime('%H:%M')}):", *tail, sep="\n   ", flush=True)
    print({n: p.returncode for n, p in procs.items()})

## Preflight (GPU 0)
Checks: 196 eligible matrices and the tied LM head, finite FP16 logits, exact restore of the FP16 checkpoint,
padding-invariant scoring, quantizer/packing round trip on real weights for 2/3/4/8 bits, AWQ folding preserves the
function, exact Wanda zero counts, exact P3 rounding, memory headroom of the Fisher pass, SDPA backends, and a short
perplexity/HellaSwag smoke run. Writes `results/env/preflight.json`.

In [ ]:
if RUN_PREFLIGHT:
    assert run("preflight", 0) == 0, "Preflight failed - read the log above before spending GPU time."

## Main runs (two GPUs in parallel)

In [ ]:
import torch
n_gpu = torch.cuda.device_count()
if n_gpu >= 2:
    wait({"gpu0_prune": launch("gpu0_prune", 0, ["family_prune"]),
          "gpu1_quant": launch("gpu1_quant", 1, ["family_quant"])})
else:
    wait({"gpu0_all": launch("gpu0_all", 0, ["family_prune", "family_quant"])})
for f in glob.glob(f"{LOGS}/*.log"):
    bad = "STEP FAILED" in open(f, errors="replace").read()
    print(f, "<-- contains a failed step, read it" if bad else "ok")

In [ ]:
run("status")
run("report")

In [ ]:
# ---------------- package the small, git-tracked outputs for download ----------------
os.makedirs(f"{TASK}/results/data_selection", exist_ok=True)
shutil.copy2(lock, f"{TASK}/results/data_selection/local_source_revisions.json")
shutil.copytree(LOGS, f"{TASK}/results/logs", dirs_exist_ok=True)
shutil.make_archive(f"{WORK}/task1_results", "zip", TASK, "results")
shutil.copy2(lock, f"{WORK}/local_source_revisions.json")
print(sorted(os.listdir(WORK)))

In [ ]:
# Inline preview
from IPython.display import Markdown, Image, display
for t in sorted(glob.glob(f"{TASK}/results/tables/*.md")):
    display(Markdown(f"### {os.path.basename(t)}\n" + open(t, encoding="utf-8").read()))
for fig in sorted(glob.glob(f"{TASK}/results/figures/*.png")):
    display(Image(fig, width=900))